In [ ]:
!wget ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
!tar -xzf sift.tar.gz

--2026-06-19 08:12:32--  ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
           => ‘sift.tar.gz’
Resolving ftp.irisa.fr (ftp.irisa.fr)... 131.254.254.45, 2001:660:7303:254::45
Connecting to ftp.irisa.fr (ftp.irisa.fr)|131.254.254.45|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /local/texmex/corpus ... done.
==> SIZE sift.tar.gz ... 168280445
==> PASV ... done.    ==> RETR sift.tar.gz ... done.
Length: 168280445 (160M) (unauthoritative)

sift.tar.gz         100%[===================>] 160.48M  24.5MB/s    in 7.3s    

2026-06-19 08:12:40 (22.0 MB/s) - ‘sift.tar.gz’ saved [168280445]



In [ ]:
# ============================================================
# ΠΕΙΡΑΜΑ ΚΛΙΜΑΚΩΣΙΜΟΤΗΤΑΣ (SCALABILITY): BASELINE vs LSH
# ============================================================

import numpy as np
import time
from pyspark.sql import SparkSession

# ==== 1. ΠΑΡΑΜΕΤΡΟΙ ====
EPSILON  = 200.0
L_TABLES = 50
K_FUNCS  = 4
W_FACTOR = 1.5
W_WIDTH  = EPSILON * W_FACTOR

CHUNK_SIZE = 1000
D_DIM    = 128
REC_LEN  = 4 + D_DIM * 4

S_LIMIT  = 1000
R_SIZES  = [1000, 10000, 100000] # Τα 3 μεγέθη που θέλουμε να δοκιμάσουμε

R_PATH   = "sift/sift_base.fvecs"
S_PATH   = "sift/sift_query.fvecs"

# ==== 2. Spark Session ====
spark = SparkSession.builder \
    .appName("LSH_Scalability_Test_Fixed") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print(f"Spark {spark.version} | Πυρήνες: {sc.defaultParallelism}")

# ==== 3. Συναρτήσεις ====
def parse_fvecs(record):
    return np.frombuffer(bytes(record[4:]), dtype=np.float32).copy()

def generate_lsh_model(d, L, K, w, seed=42):
    np.random.seed(seed)
    A = [np.random.randn(K, d).astype(np.float32) for _ in range(L)]
    B = [np.random.uniform(0, w, K).astype(np.float32) for _ in range(L)]
    return A, B

def build_bucket_index(S_dict, A, B, w, L):
    buckets = {}
    for s_id, s_vec in S_dict.items():
        for l in range(L):
            proj = np.dot(A[l], s_vec) + B[l]
            key  = (l, tuple(np.floor(proj / w).astype(np.int32).tolist()))
            buckets.setdefault(key, []).append(s_id)
    return buckets

def _process_chunk(ids, vecs, A, B, w, L, s_bkts, s_vecs, eps_sq):
    M = np.stack(vecs)
    candidates = [set() for _ in range(len(ids))]
    for l in range(L):
        projs   = M @ A[l].T + B[l]
        buckets = np.floor(projs / w).astype(np.int32)
        for i in range(len(ids)):
            key = (l, tuple(buckets[i].tolist()))
            if key in s_bkts:
                candidates[i].update(s_bkts[key])
    for i, r_id in enumerate(ids):
        r = M[i]
        for s_id in candidates[i]:
            diff = r - s_vecs[s_id]
            if float(np.dot(diff, diff)) <= eps_sq:
                yield (r_id, s_id)

def lsh_map_chunked(iterator, A_bc, B_bc, w, L, Sb_bc, Sv_bc, eps):
    A = A_bc.value; B = B_bc.value
    sb = Sb_bc.value; sv = Sv_bc.value
    eps_sq = eps ** 2
    ids, vecs = [], []
    for r_id, r_vec in iterator:
        ids.append(r_id); vecs.append(r_vec)
        if len(ids) >= CHUNK_SIZE:
            yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)
            ids, vecs = [], []
    if ids:
        yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)

# ==== 4. Φόρτωση του Query Set (Μια φορά) ====
S_dict = dict(sc.binaryRecords(S_PATH, REC_LEN)
                .map(parse_fvecs).zipWithIndex()
                .filter(lambda x: x[1] < S_LIMIT)
                .map(lambda x: (f"S_{x[1]}", x[0]))
                .collect())

# Λίστα για να αποθηκεύσουμε τα αποτελέσματα του πίνακα
results_table = []

# ==== 5. Η Λούπα της Κλιμακωσιμότητας (Ακριβώς όπως το δικό σου) ====
print(f"\nΞεκινάει η δοκιμή κλιμακωσιμότητας για R = {R_SIZES}")

for r_limit in R_SIZES:
    print(f"\n{'='*58}")
    print(f"--- Εκτέλεση για R = {r_limit:,} ---")

    # 5.1 Φόρτωση του υποσυνόλου R
    rdd_R_kv = (sc.binaryRecords(R_PATH, REC_LEN)
                  .map(parse_fvecs).zipWithIndex()
                  .filter(lambda x: x[1] < r_limit)
                  .map(lambda x: (f"R_{x[1]}", x[0])))

    # Το κάνουμε collect στον driver ΠΡΙΝ ξεκινήσει το χρονόμετρο (ακριβώς όπως το είχες)
    R_samp = dict(rdd_R_kv.collect())
    S_samp = S_dict

    # 5.2 Baseline (Η κλασική διπλή λούπα σου!)
    print("  Υπολογισμός Baseline (ground truth)...")
    t_gt = time.time()
    gt_pairs = set()
    eps_sq = EPSILON ** 2
    for r_id, r_vec in R_samp.items():
        for s_id, s_vec in S_samp.items():
            d = r_vec - s_vec
            if float(np.dot(d, d)) <= eps_sq:
                gt_pairs.add((r_id, s_id))
    t_baseline = time.time() - t_gt
    print(f"  Ground truth: {len(gt_pairs):,} ζευγάρια σε {t_baseline:.2f}s")

    # 5.3 LSH
    print("  Εκτέλεση LSH...")
    A_mat, B_mat = generate_lsh_model(D_DIM, L_TABLES, K_FUNCS, W_WIDTH)
    sb_local     = build_bucket_index(S_dict, A_mat, B_mat, W_WIDTH, L_TABLES)

    A_bc  = sc.broadcast(A_mat)
    B_bc  = sc.broadcast(B_mat)
    Sv_bc = sc.broadcast(S_dict)
    Sb_bc = sc.broadcast(sb_local)

    t0 = time.time()
    _L, _w, _e = L_TABLES, W_WIDTH, EPSILON
    matches = (rdd_R_kv
               .mapPartitions(lambda it,
                    _A=A_bc, _B=B_bc, _Sb=Sb_bc, _Sv=Sv_bc,
                    _l=_L, _ww=_w, _ee=_e:
                    lsh_map_chunked(it, _A, _B, _ww, _l, _Sb, _Sv, _ee))
               .collect())
    t_lsh = time.time() - t0

    # 5.4 Υπολογισμός Recall & Speedup
    ms     = set(matches)
    approx = {p for p in ms if p[0] in R_samp and p[1] in S_samp}
    recall = len(gt_pairs & approx) / len(gt_pairs) if gt_pairs else 1.0
    speedup = t_baseline / t_lsh if t_lsh > 0 else 0
    status = "✓" if recall >= 0.95 else "✗"

    print(f"  LSH ζευγάρια: {len(matches):,} | Χρόνος: {t_lsh:.2f}s | Recall: {recall*100:.1f}% {status}")
    print(f"  Speedup vs baseline: {speedup:.1f}x")

    results_table.append((r_limit, S_LIMIT, t_baseline, t_lsh, speedup, recall))
    A_bc.destroy(); B_bc.destroy(); Sv_bc.destroy(); Sb_bc.destroy()

# ==== 6. ΕΚΤΥΠΩΣΗ ΤΕΛΙΚΟΥ ΠΙΝΑΚΑ ====
print(f"\n{'='*75}")
print(f"| {'R Size':>9} | {'S Size':>8} | {'Baseline (s)':>12} | {'LSH Time (s)':>12} | {'Speedup':>7} | {'Recall':>7} |")
print(f"{'-'*75}")
for r_s, s_s, t_b, t_l, spd, rec in results_table:
    print(f"| {r_s:>9,} | {s_s:>8,} | {t_b:>12.2f} | {t_l:>12.2f} | {spd:>6.1f}x | {rec*100:>6.1f}% |")
print(f"{'='*75}")

sc.stop()

Spark 4.0.2 | Πυρήνες: 2

🚀 Ξεκινάει η δοκιμή κλιμακωσιμότητας για R = [1000, 10000, 100000]

--- Εκτέλεση για R = 1,000 ---
  Υπολογισμός Baseline (ground truth)...
  Ground truth: 95 ζευγάρια σε 2.31s
  Εκτέλεση LSH...
  LSH ζευγάρια: 94 | Χρόνος: 5.59s | Recall: 98.9% ✓
  Speedup vs baseline: 0.4x

--- Εκτέλεση για R = 10,000 ---
  Υπολογισμός Baseline (ground truth)...
  Ground truth: 2,154 ζευγάρια σε 27.03s
  Εκτέλεση LSH...
  LSH ζευγάρια: 2,129 | Χρόνος: 10.12s | Recall: 98.8% ✓
  Speedup vs baseline: 2.7x

--- Εκτέλεση για R = 100,000 ---
  Υπολογισμός Baseline (ground truth)...
  Ground truth: 20,477 ζευγάρια σε 261.42s
  Εκτέλεση LSH...
  LSH ζευγάρια: 20,243 | Χρόνος: 50.30s | Recall: 98.9% ✓
  Speedup vs baseline: 5.2x

|    R Size |   S Size | Baseline (s) | LSH Time (s) | Speedup |  Recall |
---------------------------------------------------------------------------
|     1,000 |    1,000 |         2.31 |         5.59 |    0.4x |   98.9% |
|    10,000 |    1,000 |       